In [1]:
import pandas as pd

df = pd.read_csv("data/restaurant_inspections.csv", low_memory=False)

print(df.shape)

(296012, 27)


296,012 = rows. Each row is one violation found during an inspection, not one restaurant. If an inspection found 4 violations, that inspection takes 4 rows.
27 = columns. These are the details recorded for each row (restaurant name, borough, inspection date, score, grade, violation code, and so on).

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 296012 entries, 0 to 296011
Data columns (total 27 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   CAMIS                  296012 non-null  int64  
 1   DBA                    296010 non-null  object 
 2   BORO                   296012 non-null  object 
 3   BUILDING               295029 non-null  object 
 4   STREET                 295977 non-null  object 
 5   ZIPCODE                292930 non-null  float64
 6   PHONE                  295914 non-null  object 
 7   CUISINE DESCRIPTION    292218 non-null  object 
 8   INSPECTION DATE        296012 non-null  object 
 9   ACTION                 292298 non-null  object 
 10  VIOLATION CODE         289568 non-null  object 
 11  VIOLATION DESCRIPTION  289568 non-null  object 
 12  CRITICAL FLAG          296012 non-null  object 
 13  SCORE                  278786 non-null  float64
 14  GRADE                  145340 non-nu

In [3]:
summary = pd.DataFrame({
    "data_type": df.dtypes,
    "non_null": df.notnull().sum(),
    "missing": df.isna().sum(),
    "missing_pct": (df.isna().sum() / len(df) * 100).round(2),
    "unique_values": df.nunique()
})

summary

,data_type,non_null,missing,missing_pct,unique_values
CAMIS,int64,296012,0,0.00,31366
DBA,object,296010,2,0.00,24628
BORO,object,296012,0,0.00,6
BUILDING,object,295029,983,0.33,8558
STREET,object,295977,35,0.01,2311
ZIPCODE,float64,292930,3082,1.04,256
PHONE,object,295914,98,0.03,27705
CUISINE DESCRIPTION,object,292218,3794,1.28,90
INSPECTION DATE,object,296012,0,0.00,1966
ACTION,object,292298,3714,1.25,5


## Step 2: Missing values

The file has 296,012 rows and 27 columns. Each row is one violation, not one inspection.

**Findings**
- GRADE (51% missing) and GRADE DATE (54% missing) are empty by design. A grade is only issued on some inspections, so we will not fill these in.
- VIOLATION CODE and DESCRIPTION are missing in 6,444 rows (2%). These rows have no violation to describe.
- ACTION and INSPECTION TYPE are each missing in 3,714 rows. This matches the 3,714 rows with the placeholder date 01/01/1900 (restaurants not yet inspected).
- SCORE is missing in 17,226 rows (6%).
- All other columns are missing less than 2% (cuisine, zipcode, census tract, neighborhood, latitude and longitude).

**Data type problems to fix**
- INSPECTION DATE, GRADE DATE and RECORD DATE are stored as text and need to become dates.
- ZIPCODE (and likely Census Tract) is stored as a number, so leading zeros can be lost. These should be text.

**Decision:** we will not drop every row with a missing value, because that would remove most of the data. We will decide column by column.

In [5]:
pd.set_option("display.max_columns", None)

df.sample(20, random_state=1)

,CAMIS,DBA,BORO,BUILDING,STREET,ZIPCODE,PHONE,CUISINE DESCRIPTION,INSPECTION DATE,ACTION,VIOLATION CODE,VIOLATION DESCRIPTION,CRITICAL FLAG,SCORE,GRADE,GRADE DATE,RECORD DATE,INSPECTION TYPE,Latitude,Longitude,Community Board,Council District,Census Tract,BIN,BBL,NTA,Location
260149,50130607,OUR BACKYARD CAFE,Queens,46-10,104 STREET,11368.0,7185659104,Bakery Products/Desserts,11/07/2024,Violations were cited in the following area(s).,10B,Anti-siphonage or back-flow prevention device ...,Not Critical,27.0,B,11/07/2024,08/26/2026,Cycle Inspection / Re-inspection,40.746230,-73.859835,404.0,21.0,41300.0,4437329.0,4.019810e+09,QN25,POINT (-73.859834714739 40.746230036241)
201461,40387861,TROOP WINGS,Manhattan,614,9 AVENUE,10036.0,2125827661,American,05/29/2025,Violations were cited in the following area(s).,04N,Filth flies or food/refuse/sewage associated w...,Critical,25.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,40.759479,-73.991849,104.0,3.0,12100.0,1024979.0,1.010340e+09,MN15,POINT (-73.99184931572 40.759479232348)
252509,50173198,NEW MAY MAY KITCHEN,Manhattan,181,CLINTON STREET,10002.0,2124201328,Chinese,11/24/2025,Violations were cited in the following area(s).,08A,Establishment is not free of harborage or cond...,Not Critical,33.0,NaN,NaN,08/26/2026,Pre-permit (Operational) / Initial Inspection,40.715431,-73.987010,103.0,1.0,1401.0,1077606.0,1.003110e+09,MN28,POINT (-73.987010122947 40.715430818244)
110071,50181493,HALAL MUNCHIES,Manhattan,1810,3 AVENUE,10029.0,9298901198,Sandwiches,05/13/2026,Establishment Closed by DOHMH. Violations were...,04L,Evidence of mice or live mice in establishment...,Critical,18.0,NaN,NaN,08/26/2026,Pre-permit (Operational) / Reopening Inspection,40.788040,-73.947258,111.0,8.0,16600.0,1087540.0,1.016288e+09,MN33,POINT (-73.947257743984 40.788040307017)
87346,41427508,KING OF TANDOOR,Brooklyn,600,FLATBUSH AVENUE,11225.0,3475336811,Indian,06/08/2026,Violations were cited in the following area(s).,04L,Evidence of mice or live mice in establishment...,Critical,23.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,40.658951,-73.960541,309.0,40.0,79802.0,3114605.0,3.050260e+09,BK60,POINT (-73.960540875347 40.658951036849)
150453,41300703,PAO DE QUEIJO & CO.,Queens,31-90,30 STREET,11106.0,7182041979,Brazilian,07/31/2024,Violations were cited in the following area(s).,02G,Cold TCS food item held above 41 °F; smoked or...,Critical,18.0,NaN,NaN,08/26/2026,Cycle Inspection / Initial Inspection,40.762516,-73.926059,401.0,22.0,7500.0,4445865.0,4.005880e+09,QN70,POINT (-73.92605928813 40.762516107761)
51560,50145402,JR SUSHI,Manhattan,119,CHAMBERS STREET,10007.0,2122338338,Japanese,02/24/2025,Violations were cited in the following area(s).,02B,Hot TCS food item not held at or above 140 °F.,Critical,26.0,Z,02/24/2025,08/26/2026,Cycle Inspection / Re-inspection,40.715225,-74.008585,101.0,1.0,2100.0,1001581.0,1.001458e+09,MN24,POINT (-74.008585340645 40.715225374917)
115142,50166044,SANDZ,Queens,234-32,MERRICK BOULEVARD,11422.0,3479442511,Caribbean,12/01/2025,Violations were cited in the following area(s).,10G,Dishwashing and ware washing: Cleaning and san...,Not Critical,21.0,N,NaN,08/26/2026,Pre-permit (Operational) / Initial Inspection,40.674307,-73.735702,413.0,31.0,63200.0,4539017.0,4.131980e+09,QN66,POINT (-73.735701520645 40.674306554243)
203430,50045623,LUCKY STAR,Brooklyn,2149,KNAPP STREET,11229.0,7183325282,Chinese,04/23/2026,Violations were cited in the following area(s).,02G,Cold TCS food item held above 41 °F; smoked or...,Critical,11.0,A,04/23/2026,08/26/2026,Cycle Inspection / Initial Inspection,40.600242,-73.934083,315.0,46.0,57000.0,3248258.0,3.088220e+09,BK17,POINT (-73.934082777456 40.60024166899)
154216,50138275,MURPHY'S BAR,Queens,45-01,23 AVENUE,11105.0,3472423786,New American,03/25/2026,Violations were cited in the following area(s).,20-06,Current letter grade or Grade Pending card not...,Not Critical,NaN,NaN,NaN,08/26/2026,Administrative Miscellaneous / Initial Insp

In [7]:
print("All restaurants:", df["CAMIS"].nunique())

inspected = df[df["INSPECTION DATE"] != "01/01/1900"]
print("Restaurants with a real inspection:", inspected["CAMIS"].nunique())

print(inspected.groupby("BORO")["CAMIS"].nunique())

All restaurants: 31366
Restaurants with a real inspection: 27652
BORO
0                   32
Bronx             2425
Brooklyn          7077
Manhattan        10850
Queens            6268
Staten Island     1000
Name: CAMIS, dtype: int64


In [ ]:
print("Restaurants with at least one score:", df[df["SCORE"].notna()]["CAMIS"].nunique())

inspections = df[df["INSPECTION DATE"] != "01/01/1900"].drop_duplicates(
    subset=["CAMIS", "INSPECTION DATE", "INSPECTION TYPE"]
)

print("Inspections:", len(inspections))
print("Inspections with a score:", inspections["SCORE"].notna().sum())
print(inspections["SCORE"].describe() 

Restaurants with at least one score: 27647
Inspections: 93292
Inspections with a score: 82956
count    82956.000000
mean        17.890388
std         14.922439
min          0.000000
25%          9.000000
50%         13.000000
75%         24.000000
max        214.000000
Name: SCORE, dtype: float64
